# Navier-Stokes PINN - flujo experimental FCN/PIKAN/cPIKAN

**Notebook base:** `NS_PIKANS_geo_BASELINE.ipynb`

Objetivo:
Comparar FCN, PIKAN-NORM y cPIKAN-NORM en Navier-Stokes 2D estacionario alrededor de cilindro.

Regla experimental:
Misma geometría, mismos puntos de frontera, mismos puntos de colocación, misma loss, mismo optimizador.

Lo único que cambia:
Arquitectura del aproximador.


## 1. Entorno, dependencias e imports



In [ ]:
!pip install torchinfo

In [ ]:
!pip install torchopt torchsummary deepxde pydoe

In [ ]:
%env DDE_BACKEND=pytorch

In [ ]:
import os
import json
import time
import random
import numpy as np
import torch
from torch import nn
from collections import OrderedDict
from dataclasses import dataclass, asdict
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torch.quasirandom import SobolEngine

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Rutas de trabajo y salida



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
root = "/content/drive/MyDrive/GradProject/NS"
model_root = root + "/NS_models"
results_root = root + "/NS_results"

os.makedirs(model_root, exist_ok=True)
os.makedirs(results_root, exist_ok=True)

print(f"Modelos: {model_root}")
print(f"Resultados: {results_root}")


## 3. Dominio fisico y constantes del problema

Esta seccion define la geometria del canal, cilindro, viscosidad, velocidad de entrada y dispositivo. No modificar entre arquitecturas si el objetivo es comparar modelos de forma justa.


In [ ]:
width = 2.2
height = 0.41
time = 20.0

In [ ]:
circle_center_x = 0.2
circle_center_y = 0.2
circle_radius = 0.05

In [ ]:
kinematic_viscocity = 0.001

In [ ]:
n_train_bc = 100

In [ ]:
U = 0.3

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 4. Geometria y condiciones de frontera

Funciones originales para mascara del cilindro, borde circular, paredes, entrada/salida y conversion a tensores. Mantener esta logica fija durante la comparacion FCN vs PIKAN/cPIKAN.


In [ ]:
def inside_circle_condition(x_orig, y_orig, r, x, y):
    return (x - x_orig) ** 2 + (y - y_orig) ** 2 < r ** 2

In [ ]:
def circle_condition(x_orig, y_orig, r, x, y):
    return (x - x_orig) ** 2 + (y - y_orig) ** 2 == r ** 2

In [ ]:
def set_circle_data(x_orig, y_orig, r, n_circle):
    theta = np.linspace(0, 2 * np.pi, n_circle)
    x_circle = x_orig + r * np.cos(theta)
    y_circle = y_orig + r * np.sin(theta)
    x_circle = np.hstack((x_circle[:, None], y_circle[:, None]))
    u_circle = np.zeros((x_circle.shape))
    return x_circle, u_circle


In [ ]:
def set_lower_and_upper_walls_boundary(x, y):
    x_up = np.hstack([x[0, :][:, None], y[0, :][:, None]])
    u_up = np.zeros((x_up.shape))

    x_low = np.hstack([x[-1, :][:, None], y[-1, :][:, None]])
    u_low = np.zeros((x_low.shape))

    return x_up, u_up, x_low, u_low


In [ ]:
def inflow(x, y):
    u_0_y = (4 * U * y * (height - y)) / (height ** 2)
    return u_0_y

In [ ]:
def set_left_wall_boundary(x, y):
    x_left = np.hstack([x[:, 0][:, None], y[:, 0][:, None]])
    u_left_y = inflow(x_left[:, 0], x_left[:, 1])
    u_left = np.hstack([u_left_y[:, None], np.zeros_like(u_left_y)[:, None]])

    return x_left, u_left

In [ ]:
def create_tensors(arrays):
  tensors = []
  for array in arrays:
    array = torch.tensor(array, dtype=torch.float32, requires_grad=True)
    tensors.append(array)
  return tensors

In [ ]:
def get_data_conditions(n, x_orig, y_orig, r, n_circle):
    # x = np.linspace(0, width, n)
    # y = np.linspace(0, height, n)
    # t = np.linspace(0, time, n)
    # x, y, t = np.meshgrid(x, y, t)
    sample = lhs(2, samples=n)
    x = sample[:, 0] * width
    y = sample[:, 1] * height
    x.sort()
    y.sort()
    print(x.shape, y.shape)
    x, y = np.meshgrid(x, y)

    x_circle, u_circle = set_circle_data(x_orig, y_orig, r, n_circle)
    x_up, u_up, x_low, u_low = set_lower_and_upper_walls_boundary(x, y)
    x_left, u_left = set_left_wall_boundary(x, y)

    x = np.vstack([x_circle, x_up, x_low, x_left])
    u = np.vstack([u_circle, u_up, u_low, u_left])
    p_column = np.zeros(shape=(u.shape[0]))
    u = np.hstack([u, p_column[:, None]])


    mask = ~inside_circle_condition(x_orig, y_orig, r, x[:, 0], x[:, 1])
    x = x[mask]
    u = u[mask]



    return x, u

## 5. Dataset de frontera y verificacion visual

Se generan y visualizan las condiciones de frontera. Estas graficas sirven como control de calidad antes del entrenamiento.


In [ ]:
n_circle = 40

In [ ]:
x_conditions, y_conditions = get_data_conditions(n_train_bc, circle_center_x, circle_center_y, circle_radius, n_circle)
x_conditions.shape

In [ ]:
plt.figure(figsize=(10, 3))
scatter = plt.scatter(x_conditions[:, 0], x_conditions[:, 1], cmap='jet', marker='x', c=y_conditions[:, 0])  # 'viridis' es un mapa de colores, pero puedes elegir otro
cbar = plt.colorbar(scatter)  # Muestra la barra de color
cbar.set_label(r'$u^{(x)}$', fontsize=15, rotation=0, labelpad=10)  # labelpad ajusta la distancia del texto a la barra
plt.title('Puntos de velocidad horizontal de fluido acorde a condiciones de frontera')
plt.xlabel('X')  # Etiqueta del eje X
plt.ylabel('Y')  # Etiqueta del eje Y

# plt.savefig(root + 'ux-NS.pdf', format='pdf')

In [ ]:
plt.figure(figsize=(10, 3))
scatter = plt.scatter(x_conditions[:, 0], x_conditions[:, 1], cmap='jet', marker='x', c=y_conditions[:, 1])  # 'viridis' es un mapa de colores, pero puedes elegir otro
cbar = plt.colorbar(scatter)  # Muestra la barra de color
cbar.set_label(r'$u^{(y)}$', fontsize=15, rotation=0, labelpad=10)  # labelpad ajusta la distancia del texto a la barra
plt.title('Puntos de velocidad vertical de fluido acorde a condiciones de frontera')
plt.xlabel('X')  # Etiqueta del eje X
plt.ylabel('Y')  # Etiqueta del eje Y
# plt.savefig(root + 'uy-NS.pdf', format='pdf')

In [ ]:
plt.figure(figsize=(10, 3))
scatter = plt.scatter(x_conditions[:, 0], x_conditions[:, 1], cmap='jet', marker='x', c=y_conditions[:, 2])  # 'viridis' es un mapa de colores, pero puedes elegir otro
cbar = plt.colorbar(scatter)  # Muestra la barra de color
cbar.set_label(r'$p$', fontsize=15, rotation=0, labelpad=10)  # labelpad ajusta la distancia del texto a la barra
plt.title('Puntos de presión de fluido acorde a condiciones de frontera')
plt.xlabel('X')  # Etiqueta del eje X
plt.ylabel('Y')  # Etiqueta del eje Y
# plt.savefig(root + 'p-NS.pdf', format='pdf')

In [ ]:
x_conditions, y_conditions = create_tensors([x_conditions, y_conditions])

## 6. Arquitecturas y matriz experimental - Ronda FCN/PIKAN/cPIKAN

Las clases PINN/FCN originales permanecen como referencia. Las arquitecturas KAN deben agregarse manualmente debajo o importarse desde un modulo externo, sin tocar residual, BCs ni entrenamiento.

### Experimentos principales

| ID | Arquitectura | Proposito | Estado |
|---|---:|---|---|
| NS-FCN-BL | `FCN(2, 50, 4, 3)` | Baseline MLP original | Implementado en el notebook base |
| NS-PIKAN-S | `PIKAN(2, 16, 2, 3, degree=3)` | PIKAN pequeno | Placeholder: implementar manualmente |
| NS-PIKAN-M | `PIKAN(2, 32, 3, 3, degree=5)` | PIKAN medio | Placeholder: implementar manualmente |
| NS-PIKAN-H | `PIKAN(2, 32, 3, 3, degree=7)` | PIKAN grado alto | Placeholder: implementar manualmente |
| NS-cPIKAN-S | `cPIKAN(2, 16, 2, 3, degree=3)` | cPIKAN pequeno | Placeholder: implementar manualmente |
| NS-cPIKAN-M | `cPIKAN(2, 32, 3, 3, degree=5)` | cPIKAN medio | Placeholder: implementar manualmente |
| NS-cPIKAN-H | `cPIKAN(2, 32, 3, 3, degree=7)` | cPIKAN grado alto | Placeholder: implementar manualmente |

### Punto unico de cambio

La celda de `model = ...` es el selector del experimento. Para cada corrida, reemplazar solo esa instancia por la arquitectura deseada y actualizar el nombre del experimento/checkpoint.


In [ ]:
class NavierStokesPINN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, activation_fn=nn.Tanh()):
        super(NavierStokesPINN, self).__init__()
        input_layer = nn.Linear(input_size, hidden_size)
        hidden_layers = []
        for _ in range(num_layers):
            hidden_layers.append(nn.Linear(hidden_size, hidden_size))
            hidden_layers.append(activation_fn)
        output_layer = nn.Linear(hidden_size, output_size)
        self.net = nn.Sequential(input_layer, *hidden_layers, output_layer)

    def forward(self, x):
        return self.net(x)


In [ ]:
class FCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(FCN, self).__init__()
        activation = nn.Tanh

        layers = OrderedDict([
            ('input', nn.Linear(input_size, hidden_size)),
            ('act0', activation())
        ])

        # Añadiendo capas ocultas
        for i in range(num_layers):
            layers[f'hidden_{i}'] = nn.Linear(hidden_size, hidden_size)
            layers[f'act_{i}'] = activation()

        layers['output'] = nn.Linear(hidden_size, output_size)

        self.net = nn.Sequential(layers)

    def forward(self, x):
        return self.net(x)

### B-spline KAN Layer

In [ ]:
def normalize_xy(data):
    x = data[:, 0:1] / width
    y = data[:, 1:2] / height
    return torch.cat([x, y], dim=1) * 2.0 - 1.0

In [ ]:
class BSplineKANLayer(nn.Module):
    def __init__(self, in_features, out_features, degree=3,
                 n_knots=5, grid_range=(-1, 1)):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.degree       = degree
        self.n_knots      = n_knots

        # Para open/clamped B-splines:
        # número de bases = len(knots) - degree - 1
        knots_interior = torch.linspace(grid_range[0], grid_range[1], n_knots + 1)
        knots = torch.cat([
            knots_interior[0].repeat(degree),
            knots_interior,
            knots_interior[-1].repeat(degree)
        ])

        self.register_buffer("knots", knots)

        self.n_basis = len(knots) - degree - 1

        self.coeff = nn.Parameter(
            torch.randn(in_features, out_features, self.n_basis) * 0.1
        )
        self.w_base = nn.Parameter(torch.ones(in_features, out_features))
        self.w_spline = nn.Parameter(torch.ones(in_features, out_features))

    def b_spline_basis(self, x):
        """
        Cox-de Boor vectorizado.
        x: (N, in_features)
        returns: (N, in_features, n_basis)
        """
        k = self.knots
        x_e = x.unsqueeze(-1)

        # Grado 0: len(k)-1 funciones
        B = ((x_e >= k[:-1].view(1, 1, -1)) &
             (x_e <  k[1:].view(1, 1, -1))).float()

        # Incluir el extremo derecho para evitar que x == max quede todo cero
        B = torch.where(
            (x_e == k[-1]) & (torch.arange(B.shape[-1], device=x.device).view(1, 1, -1) == B.shape[-1] - 1),
            torch.ones_like(B),
            B
        )

        # Recursión Cox-de Boor
        for p in range(1, self.degree + 1):
            n_prev = B.shape[-1]
            n_curr = n_prev - 1

            B_left = B[..., :n_curr]
            B_right = B[..., 1:n_curr + 1]

            t_i = k[:n_curr].view(1, 1, -1)
            t_ip = k[p:p + n_curr].view(1, 1, -1)
            t_i1 = k[1:n_curr + 1].view(1, 1, -1)
            t_ip1 = k[p + 1:p + 1 + n_curr].view(1, 1, -1)

            denom1 = (t_ip - t_i).clamp(min=1e-8)
            denom2 = (t_ip1 - t_i1).clamp(min=1e-8)

            term1 = ((x_e - t_i) / denom1) * B_left
            term2 = ((t_ip1 - x_e) / denom2) * B_right

            B = term1 + term2

        return B

    def forward(self, x):
        base = torch.nn.functional.silu(x)
        B = self.b_spline_basis(x)

        spline_out = torch.einsum("nib,iob->nio", B, self.coeff)

        out = (
            self.w_base * base.unsqueeze(2)
            + self.w_spline * spline_out
        ).sum(dim=1)

        return out

In [ ]:
class PIKAN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers,
                 output_size, degree=3, n_knots=5):
        super().__init__()
        layers = []
        dims   = [input_size] + [hidden_size] * num_layers + [output_size]
        for i in range(len(dims) - 1):
            layers.append(
                BSplineKANLayer(dims[i], dims[i+1],
                                degree=degree, n_knots=n_knots)
            )
        self.layers = nn.ModuleList(layers)

        # Rangos reales del dominio para normalizar input
        self.register_buffer('x_min', torch.tensor([0.0,    0.0   ]))
        self.register_buffer('x_max', torch.tensor([width,  height]))

    def forward(self, x):
        # Normalizar coordenadas físicas [0,2.2]×[0,0.41] → [-1,1]²
        x = 2.0 * (x - self.x_min) / (self.x_max - self.x_min) - 1.0
        for layer in self.layers:
            x = layer(x)
        return x

### Chebyshev KAN Layer (cPIKAN)

In [ ]:
class ChebyshevKANLayer(nn.Module):
    """
    Capa KAN con base de polinomios de Chebyshev T_k(x).
    Mas rapida que B-spline: evaluacion O(degree) sin recursion de grilla.
    El input se normaliza a [-1,1] antes de evaluar los polinomios.

    Parametros:
        in_features  : numero de entradas
        out_features : numero de salidas
        degree       : grado maximo del polinomio (numero de terminos = degree+1)
    """
    def __init__(self, in_features, out_features, degree=3):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.degree       = degree

        # Coeficientes: (in, out, degree+1)
        self.coeff  = nn.Parameter(
            torch.randn(in_features, out_features, degree + 1) * 0.1
        )
        self.w_base = nn.Parameter(torch.ones(in_features, out_features))

    def chebyshev_basis(self, x):
        """
        Evalua T_0..T_degree sobre x usando la relacion de recurrencia:
            T_0 = 1,  T_1 = x,  T_k = 2x*T_{k-1} - T_{k-2}
        x: (N, in_features)  ->  retorna (N, in_features, degree+1)
        """
        # Normalizar a [-1, 1] con tanh para robustez numerica
        x_norm = torch.tanh(x)

        polys = [torch.ones_like(x_norm),   # T_0
                 x_norm]                     # T_1
        for k in range(2, self.degree + 1):
            polys.append(2 * x_norm * polys[-1] - polys[-2])

        # Stack: (degree+1, N, in) -> (N, in, degree+1)
        return torch.stack(polys, dim=-1)

    def forward(self, x):
        """
        x: (N, in_features)
        retorna: (N, out_features)
        """
        base = torch.nn.functional.silu(x)          # (N, in)
        T    = self.chebyshev_basis(x)               # (N, in, degree+1)

        # Combinar: einsum sobre degree+1
        cheb_out = torch.einsum('nid,iod->nio', T, self.coeff)  # (N,in,out)

        out = (self.w_base * base.unsqueeze(2) + cheb_out).sum(dim=1)
        return out

In [ ]:
class cPIKAN(nn.Module):
    """
    cPIKAN: apila ChebyshevKANLayers.
    Interfaz identica a FCN y PIKAN.

    Ejemplo: cPIKAN(2, 16, 2, 3, degree=3)
    """
    def __init__(self, input_size, hidden_size, num_layers,
                 output_size, degree=3):
        super().__init__()
        layers = []
        dims   = [input_size] + [hidden_size] * num_layers + [output_size]
        for i in range(len(dims) - 1):
            layers.append(
                ChebyshevKANLayer(dims[i], dims[i+1], degree=degree)
            )
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        x = 2.0 * (x - self.x_min) / (self.x_max - self.x_min) - 1.0
        for layer in self.layers:
            x = layer(x)
        return x

## 7. Residual Navier-Stokes y funciones de perdida

Mantener esta seccion sin cambios entre arquitecturas. Aqui esta la definicion experimental comun: residual fisico, perdida de condiciones de frontera y ponderacion de terminos.


In [ ]:
loss_fn = nn.MSELoss()

In [ ]:
# def physics_loss(x_coll, y_coll):
#     x_coll = torch.hstack((x_coll, y_coll))
#     # print(x_coll)
#     y_hat = model(x_coll)

#     u_pred = y_hat[:, 0:1]
#     v_pred = y_hat[:, 1:2]
#     p_pred = y_hat[:, 2:3]

#     x = x_coll[:, 0].unsqueeze(1)
#     y = x_coll[:, 1].unsqueeze(1)

#     ux = dde.grad.jacobian(y_hat, x_coll, i=0, j=0)
#     uy = dde.grad.jacobian(y_hat, x_coll, i=0, j=1)

#     vx = dde.grad.jacobian(y_hat, x_coll, i=1, j=0)
#     vy = dde.grad.jacobian(y_hat, x_coll, i=1, j=1)

#     px = dde.grad.jacobian(y_hat, x_coll, i=2, j=0)
#     py = dde.grad.jacobian(y_hat, x_coll, i=2, j=1)

#     uxx = dde.grad.hessian(y_hat, x_coll, i=0, j=0)
#     uyy = dde.grad.hessian(y_hat, x_coll, i=0, j=1)

#     vxx = dde.grad.hessian(y_hat, x_coll, i=1, j=0)
#     vyy = dde.grad.hessian(y_hat, x_coll, i=1, j=1)

#     # Ecuaciones de Navier-Stokes para fluidos incompresibles
#     ns_x = -kinematic_viscocity * (uxx + uyy) + u_pred * ux + v_pred * uy + px
#     ns_y = -kinematic_viscocity * (vxx + vyy) + u_pred * vx + v_pred * vy + py

#     # Ecuación de continuidad para flujo incompresible
#     continuity = ux + vy

#     loss = loss_fn(ns_x, torch.zeros_like(ns_x)) + \
#         loss_fn(ns_y, torch.zeros_like(ns_y)) + \
#         loss_fn(continuity, torch.zeros_like(continuity))

#     return loss


In [ ]:
def physics_loss(x_coll, y_coll):
    # Asume que x_coll es un tensor de PyTorch con requires_grad=True
    data = torch.hstack((x_coll, y_coll))
    y_hat = model(data)
    u_pred = y_hat[:, 0:1]
    v_pred = y_hat[:, 1:2]
    p_pred = y_hat[:, 2:3]

    ux = torch.autograd.grad(u_pred, x_coll, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    uy = torch.autograd.grad(u_pred, y_coll, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    vx = torch.autograd.grad(v_pred, x_coll, grad_outputs=torch.ones_like(v_pred), create_graph=True)[0]
    vy = torch.autograd.grad(v_pred, y_coll, grad_outputs=torch.ones_like(v_pred), create_graph=True)[0]
    px = torch.autograd.grad(p_pred, x_coll, grad_outputs=torch.ones_like(p_pred), create_graph=True)[0]
    py = torch.autograd.grad(p_pred, y_coll, grad_outputs=torch.ones_like(p_pred), create_graph=True)[0]

    uxx = torch.autograd.grad(ux, x_coll, grad_outputs=torch.ones_like(ux), create_graph=True)[0]
    uyy = torch.autograd.grad(uy, y_coll, grad_outputs=torch.ones_like(uy), create_graph=True)[0]
    vxx = torch.autograd.grad(vx, x_coll, grad_outputs=torch.ones_like(vx), create_graph=True)[0]
    vyy = torch.autograd.grad(vy, y_coll, grad_outputs=torch.ones_like(vy), create_graph=True)[0]


    # Ecuaciones de Navier-Stokes para fluidos incompresibles
    ns_x = -kinematic_viscocity * (uxx + uyy) + u_pred * ux + v_pred * uy + px
    ns_y = -kinematic_viscocity * (vxx + vyy) + u_pred * vx + v_pred * vy + py

    # Ecuación de continuidad para flujo incompresible
    continuity = ux + vy

    # print(loss_fn(ns_x, torch.zeros_like(ns_x)), loss_fn(ns_y, torch.zeros_like(ns_y)), loss_fn(continuity, torch.zeros_like(continuity)))
    # print(ns_x)

    # Suma los componentes de la pérdida
    loss = torch.mean(ns_x ** 2) + torch.mean(ns_y ** 2) + torch.mean(continuity ** 2)
    return loss


In [ ]:
def conditions_loss(x, y):
    y_hat = model(x)

    u_pred = y_hat[:, 0:1]
    v_pred = y_hat[:, 1:2]
    p_pred = y_hat[:, 2:3]

    u_real = y[:, 0:1]
    v_real = y[:, 1:2]
    p_real = y[:, 2:3]

    return loss_fn(u_pred, u_real) + loss_fn(v_pred, v_real)

In [ ]:
# def loss_func(x_bc, y_bc, x_collocation):
#     physics_l = physics_loss(x_collocation)
#     conditions_l = conditions_loss(x_bc, y_bc)

#     # print(physics_l, conditions_l)

#     loss = 10 * physics_l + conditions_l
#     return loss

In [ ]:
def grad_global_norm(loss, model):
    grads = torch.autograd.grad(
        loss,
        [p for p in model.parameters() if p.requires_grad],
        retain_graph=True,
        create_graph=False,
        allow_unused=True
    )

    norms = []
    for g in grads:
        if g is not None:
            norms.append(torch.norm(g.detach()))

    if len(norms) == 0:
        return torch.tensor(1.0, device=device)

    return torch.norm(torch.stack(norms))


def update_loss_weights(physics_l, conditions_l, loss_weights, alpha=0.9, clamp_min=0.1, clamp_max=10.0):
    """
    Dynamic loss weighting based on global gradient norms.

    Balances:
    - residual Navier-Stokes loss
    - boundary condition loss
    """

    grad_r_norm = grad_global_norm(physics_l, model)
    grad_bc_norm = grad_global_norm(conditions_l, model)

    eps = 1e-8
    total_norm = grad_r_norm + grad_bc_norm + eps

    lambda_r_hat = total_norm / (grad_r_norm + eps)
    lambda_bc_hat = total_norm / (grad_bc_norm + eps)

    lambda_r = loss_weights['residual']
    lambda_bc = loss_weights['bc']

    if not torch.is_tensor(lambda_r):
        lambda_r = torch.tensor(float(lambda_r), device=device)

    if not torch.is_tensor(lambda_bc):
        lambda_bc = torch.tensor(float(lambda_bc), device=device)

    lambda_r = alpha * lambda_r + (1.0 - alpha) * lambda_r_hat
    lambda_bc = alpha * lambda_bc + (1.0 - alpha) * lambda_bc_hat

    lambda_r = torch.clamp(lambda_r.detach(), clamp_min, clamp_max)
    lambda_bc = torch.clamp(lambda_bc.detach(), clamp_min, clamp_max)

    return {
        'residual': lambda_r,
        'bc': lambda_bc
    }

In [ ]:
def loss_func(
    x_bc,
    y_bc,
    x_collocation,
    y_collocation,
    loss_weights,
    n,
    f,
    auto_w=False
):
    physics_l = physics_loss(x_collocation, y_collocation)
    conditions_l = conditions_loss(x_bc, y_bc)

    if n > 0 and n % f == 0 and auto_w:
        loss_weights = update_loss_weights(
            physics_l,
            conditions_l,
            loss_weights,
            alpha=0.9,
            clamp_min=0.1,
            clamp_max=10.0
        )

    w_physics = loss_weights['residual']
    w_conditions = loss_weights['bc']

    loss = w_physics * physics_l + w_conditions * conditions_l

    return loss, loss_weights

## 8. Puntos de colocacion y dataloaders

Mantener el muestreo y los tamanos de batch constantes para comparar arquitecturas. Si se cambia `n_train_coll` o `n_train_bc`, registrar una nueva ronda experimental.


In [ ]:
from torch.quasirandom import SobolEngine

def get_collocation_points(n_train, x_orig, y_orig, r, seed=42):
    # Sobol quasi-random — mejor cobertura que LHS, reproducible con seed
    sobol  = SobolEngine(dimension=2, scramble=True, seed=seed)
    sample = sobol.draw(n_train).numpy()          # (n_train, 2) en [0,1]²

    x = sample[:, 0] * width
    y = sample[:, 1] * height

    # Sin meshgrid — los n_train puntos se usan directamente (fix bug original)
    x_train = np.hstack((x[:, None], y[:, None])) # (n_train, 2)

    x_circle, _ = set_circle_data(x_orig, y_orig, r, 50)

    mask    = ~inside_circle_condition(x_orig, y_orig, r,
                                       x_train[:, 0], x_train[:, 1])
    x_train = x_train[mask]
    x_train = np.vstack((x_train, x_circle))

    x_train, y_train = x_train[:, 0], x_train[:, 1]
    x_train = x_train.reshape(-1, 1)
    y_train = y_train.reshape(-1, 1)

    plt.figure(figsize=(10, 3))
    plt.scatter(x_train, y_train, s=8, c='steelblue')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(f'Puntos de colocación — Sobol (n={len(x_train)}, seed={seed})')
    plt.tight_layout()
    plt.show()

    print(f"  Puntos de colocación totales : {len(x_train)}")
    print(f"  Sobol base                   : {n_train}")
    print(f"  Puntos del cilindro          : 50")
    print(f"  Puntos removidos (interior)  : {n_train - mask.sum()}")

    x_train = torch.tensor(x_train, dtype=torch.float32, requires_grad=True)
    y_train = torch.tensor(y_train, dtype=torch.float32, requires_grad=True)

    return x_train, y_train

In [ ]:
# def get_boundary_points_batch(n_batch):
#   x_bc =

In [ ]:
n_train_coll = 1500

In [ ]:
x_collocation, y_collocation = get_collocation_points(n_train_coll, circle_center_x, circle_center_y, circle_radius)

In [ ]:
x_collocation.shape

In [ ]:
dataloader_collocation = DataLoader(TensorDataset(x_collocation, y_collocation), batch_size=32, shuffle=True)
dataloader_conditions = DataLoader(TensorDataset(x_conditions, y_conditions), batch_size=32, shuffle=True)

In [ ]:
def get_test_grid(nx=220, ny=80):
    x_lin = np.linspace(0, width, nx)
    y_lin = np.linspace(0, height, ny)
    xx, yy = np.meshgrid(x_lin, y_lin)
    x_flat = xx.flatten()[:, None]
    y_flat = yy.flatten()[:, None]
    mask = ~inside_circle_condition(
        circle_center_x, circle_center_y, circle_radius,
        x_flat[:, 0], y_flat[:, 0]
    )
    x_t = torch.tensor(x_flat[mask], dtype=torch.float32)
    y_t = torch.tensor(y_flat[mask], dtype=torch.float32)
    print(f"  Grid de test: {x_t.shape[0]:,} puntos ({nx}×{ny} - cilindro)")
    return x_t, y_t

## 9. Anti-desconexion de Colab

Ejecutar esta celda al inicio de cada sesion larga, despues del setup y antes del entrenamiento. Ayuda contra desconexion por inactividad del navegador.

In [ ]:
%%javascript
function KeepAlive() {
    document.querySelector(
        '#top-toolbar > colab-connect-button'
    )?.shadowRoot?.querySelector('#connect')?.click();
}
setInterval(KeepAlive, 60000);

## 10. Seleccion de modelo

### Opcion A - Dropdown interactivo

Usar esta opcion para trabajo manual en Colab: seleccionas un modelo y ejecutas la celda de instanciacion.

### Opcion B - Configuracion reproducible

Para experimentos mas adaptables que un dropdown, usa `EXPERIMENT_ID` como fuente de verdad. Esto es mejor cuando quieras automatizar corridas, repetir resultados, ejecutar con Papermill/Colab Pro, o lanzar una arquitectura sin depender de interaccion visual.

El notebook deja ambas rutas disponibles: por defecto usa el dropdown; si quieres modo reproducible, cambia `USE_DROPDOWN = False` y define `EXPERIMENT_ID`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

model_selector = widgets.Dropdown(
    options=[
        ('FCN baseline - PINN MLP original', 'FCN'),
        ('PIKAN average-case - B-spline degree=3', 'PIKAN-AVG'),
        ('cPIKAN average-case - Chebyshev degree=3', 'cPIKAN-AVG'),
    ],
    value='FCN',
    description='Modelo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='480px')
)

display(model_selector)

In [ ]:
USE_DROPDOWN = False
EXPERIMENT_ID = 'PIKAN-AVG'

model_configs = {
    'FCN': {
        'factory': lambda: FCN(2, 50, 4, 3),
        'label': 'FCN baseline',
        'family': 'FCN',
        'hidden': 50,
        'layers': 4,
        'degree': None,
        'notes': 'Baseline PINN/MLP original'
    },

    'PIKAN-AVG': {
        'factory': lambda: PIKAN(2, 16, 2, 3, degree=3),
        'label': 'PIKAN average-case',
        'family': 'PIKAN',
        'hidden': 16,
        'layers': 2,
        'degree': 3,
        'notes': 'B-spline KAN razonable para toma de contacto'
    },

    'cPIKAN-AVG': {
        'factory': lambda: cPIKAN(2, 16, 2, 3, degree=3),
        'label': 'cPIKAN average-case',
        'family': 'cPIKAN',
        'hidden': 16,
        'layers': 2,
        'degree': 3,
        'notes': 'Chebyshev KAN razonable para toma de contacto'
    },
}

## 11. Instanciacion del modelo

Ejecutar despues de elegir el dropdown o definir `EXPERIMENT_ID`. Esta celda crea el modelo, fija `experiment_name`, muestra el resumen y deja listo el objeto para crear optimizadores.

In [ ]:
experiment_name = model_selector.value if USE_DROPDOWN else EXPERIMENT_ID
experiment_config = model_configs[experiment_name]

model = experiment_config['factory']()
model.to(device)

print(f"Modelo seleccionado: {experiment_name} - {experiment_config['label']}")
print(f"Familia: {experiment_config['family']}")
print(f"Hidden: {experiment_config['hidden']} | Layers: {experiment_config['layers']} | Degree: {experiment_config['degree']}")
print(f"Notas: {experiment_config['notes']}")

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)

print("\nParámetros")
print(f"  Total:      {total_params:,}")
print(f"  Trainable:  {trainable_params:,}")

print("\nParámetros por módulo hoja:")
for name, module in model.named_modules():
    if len(list(module.children())) == 0:
        params = sum(p.numel() for p in module.parameters())
        if params > 0:
            print(f"  {name:<35} {module.__class__.__name__:<25} {params:,}")

# Sanity check de forward
with torch.no_grad():
    x_dummy = torch.randn(4, 2).to(device)
    y_dummy = model(x_dummy)

print("\nSanity check")
print(f"  Input dummy:  {x_dummy.shape}")
print(f"  Output dummy: {y_dummy.shape}")

try:
    from torchinfo import summary as torchinfo_summary

    print("\nTorchinfo summary")
    torchinfo_summary(
        model,
        input_size=(1, 2),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        depth=3
    )
except Exception as e:
    print("\nTorchinfo no disponible o falló. Conteo manual usado correctamente.")
    print(f"Detalle: {type(e).__name__}: {e}")

## 12. Entrenamiento con checkpoints automaticos

Crea los optimizadores despues de instanciar el modelo. Durante Adam se guarda un checkpoint cada `checkpoint_every` epocas en `model_root`. Si Colab muere, repite setup, selecciona el mismo modelo, instancia, crea optimizadores y ejecuta `load_checkpoint(experiment_name)` antes de entrenar.

In [ ]:
epochs_ADAM = 2000
epochs_LBFGS = 1000

total_epochs = epochs_ADAM + epochs_LBFGS

In [ ]:
optimizer_ADAM = torch.optim.Adam(params=model.parameters(), lr=0.001)
optimizer_LBFGS = torch.optim.LBFGS(params=model.parameters(), lr=0.1)

In [ ]:
n_batch = 100

In [ ]:
x_conditions = x_conditions.to(device)
y_conditions = y_conditions.to(device)
x_collocation = x_collocation.to(device)
y_collocation = y_collocation.to(device)

In [ ]:
weights = {
    'residual': 1.0,
    'bc': 1.0
}

In [ ]:
f = 500

In [ ]:
loss_list = []
epoch_start = 0
checkpoint_every = 200
RESUME_IF_EXISTS = False
SKIP_IF_DONE = True
AUTO_WEIGHT_ADAM = True
AUTO_WEIGHT_LBFGS = False

In [ ]:
checkpoint_path = os.path.join(model_root, f"{experiment_name}_checkpoint.pth")
final_model_path = os.path.join(model_root, f"{experiment_name}_final.pth")

optimizer_ADAM = torch.optim.Adam(params=model.parameters(), lr=0.001)
optimizer_LBFGS = torch.optim.LBFGS(
    params=model.parameters(),
    lr=0.1,
    max_iter=20,
    max_eval=None,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    history_size=100,
    line_search_fn="strong_wolfe"
)

x_conditions = x_conditions.to(device)
y_conditions = y_conditions.to(device)
x_collocation = x_collocation.to(device)
y_collocation = y_collocation.to(device)

weights = {
    'residual': torch.tensor(1.0, device=device),
    'bc': torch.tensor(1.0, device=device)
}

f = 500
loss_list = []
epoch_start = 0
phase_start = "adam"

def _weights_to_device(weights_dict):
    fixed = {}
    for k, v in weights_dict.items():
        if torch.is_tensor(v):
            fixed[k] = v.to(device)
        else:
            fixed[k] = torch.tensor(float(v), device=device)
    return fixed

def save_checkpoint(epoch, phase):
    ckpt = {
        'epoch': epoch,
        'phase': phase,
        'model_state': model.state_dict(),
        'optimizer_adam_state': optimizer_ADAM.state_dict(),
        'optimizer_lbfgs_state': optimizer_LBFGS.state_dict(),
        'loss_list': loss_list,
        'weights': {
            k: v.detach().cpu() if torch.is_tensor(v) else v
            for k, v in weights.items()
        },
        'experiment_name': experiment_name,
        'experiment_config': {
            k: v for k, v in experiment_config.items()
            if k != 'factory'
        },
        'epochs_ADAM': epochs_ADAM,
        'epochs_LBFGS': epochs_LBFGS,
    }

    torch.save(ckpt, checkpoint_path)

def load_checkpoint_if_exists():
    global epoch_start, loss_list, weights, phase_start

    if not RESUME_IF_EXISTS:
        print("Resume desactivado. Entrenamiento desde cero.")
        return

    if not os.path.exists(checkpoint_path):
        print("No hay checkpoint previo. Entrenamiento desde cero.")
        return

    ckpt = torch.load(checkpoint_path, map_location=device)

    saved_name = ckpt.get('experiment_name', None)
    if saved_name != experiment_name:
        raise ValueError(
            f"Checkpoint incompatible: contiene {saved_name}, "
            f"pero el experimento actual es {experiment_name}."
        )

    model.load_state_dict(ckpt['model_state'])

    if 'optimizer_adam_state' in ckpt:
        optimizer_ADAM.load_state_dict(ckpt['optimizer_adam_state'])

    if 'optimizer_lbfgs_state' in ckpt:
        optimizer_LBFGS.load_state_dict(ckpt['optimizer_lbfgs_state'])

    epoch_start = int(ckpt.get('epoch', -1)) + 1
    loss_list = ckpt.get('loss_list', [])
    weights = _weights_to_device(ckpt.get('weights', weights))
    phase_start = ckpt.get('phase', 'adam')

    print(f"Checkpoint encontrado: {checkpoint_path}")
    print(f"Retomando {experiment_name} desde época global {epoch_start}")
    print(f"Fase guardada: {phase_start}")

    if len(loss_list) > 0:
        print(f"Última loss registrada: {loss_list[-1]:.4e}")

load_checkpoint_if_exists()

print("\nConfiguración de entrenamiento")
print(f"  Adam epochs:     {epochs_ADAM}")
print(f"  L-BFGS epochs:   {epochs_LBFGS}")
print(f"  Epoch start:     {epoch_start}")
print(f"  Checkpoint path: {checkpoint_path}")
print(f"  Collocation:     {x_collocation.shape[0]:,}")
print(f"  Boundary:        {x_conditions.shape[0]:,}")

if x_collocation.shape[0] > 5000:
    print("⚠️ Muchos puntos de colocación. Puede dominar el costo de autograd.")

In [ ]:
## 9.1 Loop de entrenamiento con resume y progreso

import time
from tqdm.notebook import tqdm
from IPython.display import display
import ipywidgets as widgets

t_start = time.time()

print(f"Collocation points: {x_collocation.shape[0]:,}")
if x_collocation.shape[0] > 5000:
    print("⚠️  Demasiados puntos — meshgrid bug activo. Corregir antes de continuar.")

# ── Adam ──────────────────────────────────────────────────────────
if epoch_start < epochs_ADAM:
    pbar = tqdm(
        range(epoch_start, epochs_ADAM),
        desc=f"Adam [{experiment_name}]",
        ncols=900,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}",
        leave=True
    )

    for epoch in pbar:
        model.train()
        optimizer_ADAM.zero_grad()

        loss, weights = loss_func(
            x_conditions,
            y_conditions,
            x_collocation,
            y_collocation,
            weights,
            epoch,
            f,
            True
        )

        loss.backward()
        optimizer_ADAM.step()

        loss_value = float(loss.detach().cpu())
        loss_list.append(loss_value)

        if epoch % 10 == 0:
            pbar.set_postfix(ordered_dict={
                'loss': f"{loss_value:.3e}",
                'w_res': f"{weights['residual'].detach().cpu().item():.2f}",
                'w_bc': f"{weights['bc'].detach().cpu().item():.2f}",
                'ep': epoch
            })

        if epoch > 0 and epoch % checkpoint_every == 0:
            save_checkpoint(epoch, phase="adam")
            pbar.write(f"✓ checkpoint Adam @ época {epoch} | loss={loss_value:.3e}")

    save_checkpoint(epochs_ADAM - 1, phase="adam_done")
    print("Adam terminado.")
else:
    print("Adam ya fue completado según checkpoint.")

# ── L-BFGS ────────────────────────────────────────────────────────
lbfgs_epoch_start = max(0, epoch_start - epochs_ADAM)

if lbfgs_epoch_start < epochs_LBFGS:
    pbar = tqdm(
        range(lbfgs_epoch_start, epochs_LBFGS),
        desc=f"L-BFGS [{experiment_name}]",
        ncols=900,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}",
        leave=True
    )

    for lbfgs_epoch in pbar:
        global_epoch = epochs_ADAM + lbfgs_epoch

        def closure():
            optimizer_LBFGS.zero_grad()

            loss, _ = loss_func(
                x_conditions,
                y_conditions,
                x_collocation,
                y_collocation,
                weights,
                global_epoch,
                f,
                False
            )

            loss.backward()
            return loss

        model.train()
        loss = optimizer_LBFGS.step(closure)

        loss_value = float(loss.detach().cpu())
        loss_list.append(loss_value)

        if lbfgs_epoch % 10 == 0:
            pbar.set_postfix(ordered_dict={
                'loss': f"{loss_value:.3e}",
                'ep': global_epoch
            })

        if lbfgs_epoch > 0 and lbfgs_epoch % checkpoint_every == 0:
            save_checkpoint(global_epoch, phase="lbfgs")
            pbar.write(f"✓ checkpoint L-BFGS @ época global {global_epoch} | loss={loss_value:.3e}")

    save_checkpoint(total_epochs - 1, phase="done")
    print("L-BFGS terminado.")
else:
    print("L-BFGS ya fue completado según checkpoint.")

# Guardado final liviano
torch.save({
    'experiment_name': experiment_name,
    'experiment_config': {
        k: v for k, v in experiment_config.items()
        if k != 'factory'
    },
    'model_state': model.state_dict(),
    'loss_list': loss_list,
    'weights': {
        k: v.detach().cpu() if torch.is_tensor(v) else v
        for k, v in weights.items()
    },
}, final_model_path)

elapsed = time.time() - t_start

print("\nEntrenamiento finalizado")
print(f"  Experimento: {experiment_name}")
print(f"  Tiempo total esta sesión: {elapsed / 60:.2f} min")
print(f"  Checkpoint: {checkpoint_path}")
print(f"  Modelo final: {final_model_path}")

if len(loss_list) > 0:
    print(f"  Última loss: {loss_list[-1]:.4e}")

In [ ]:
weights

## 13. Guardado de modelos y artefactos en Drive

Ejecutar esta seccion despues del entrenamiento. Guarda el modelo completo, el `state_dict`, la historia de perdida y metadata suficiente para comparar corridas despues.

In [ ]:
final_model_path = model_root + f"/{experiment_name}_final_model.pth"
state_dict_path = model_root + f"/{experiment_name}_final_state_dict.pth"
metadata_path = results_root + f"/{experiment_name}_metadata.json"
loss_path = results_root + f"/{experiment_name}_loss.npy"

metadata = {
    'experiment_name': experiment_name,
    'experiment_config': {k: v for k, v in experiment_config.items() if k != 'factory'},
    'epochs_ADAM': epochs_ADAM,
    'epochs_LBFGS': epochs_LBFGS,
    'final_loss': float(loss_list[-1]) if len(loss_list) else None,
    'num_loss_points': len(loss_list),
    'checkpoint_every': checkpoint_every,
    'model_root': model_root,
    'results_root': results_root,
}

torch.save(model, final_model_path)
torch.save(model.state_dict(), state_dict_path)
np.save(loss_path, np.array(loss_list))

with open(metadata_path, 'w') as f_json:
    json.dump(metadata, f_json, indent=2)

print(f"Modelo completo: {final_model_path}")
print(f"State dict: {state_dict_path}")
print(f"Metadata: {metadata_path}")
print(f"Loss history: {loss_path}")

## 14. Diagnóstico por corrida — curva de loss del experimento activo




In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.log(loss_list), color='steelblue', lw=1.5, label=experiment_name)
ax.axvline(x=epochs_ADAM, color='gray', linestyle='--', lw=1,
           alpha=0.7, label='Adam → L-BFGS')
ax.set_xlabel('Épocas')
ax.set_ylabel('log(loss)')
ax.set_title(f'Curva de pérdida — {experiment_name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(root + f'loss_{experiment_name}.pdf', format='pdf')
plt.show()

print(f"  Loss inicial  : {loss_list[0]:.4e}")
print(f"  Loss final    : {loss_list[-1]:.4e}")
print(f"  Épocas totales: {len(loss_list)}")
print(f"  Parámetros    : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
with torch.no_grad():
    bc_l = conditions_loss(x_conditions, y_conditions)

phys_l = physics_loss(x_collocation, y_collocation)

print(f"Physics loss: {phys_l.item():.4e}")
print(f"BC loss:      {bc_l.item():.4e}")

## 14.1 Comparación acumulada — todas las corridas guardadas


In [ ]:
import os

checkpoints_available = [
    f for f in os.listdir(root)
    if f.endswith('_checkpoint.pth')
]

if len(checkpoints_available) < 2:
    print("Menos de 2 checkpoints disponibles. Correr más experimentos primero.")
else:
    fig, ax = plt.subplots(figsize=(14, 5))

    summary_rows = []
    for fname in sorted(checkpoints_available):
        ckpt = torch.load(root + fname, map_location='cpu')
        name      = ckpt['experiment_name']
        loss_curve = ckpt['loss_list']
        final_loss = loss_curve[-1]

        ax.plot(np.log(loss_curve), lw=1.5, label=f"{name} (final={final_loss:.2e})")
        summary_rows.append((name, len(loss_curve), final_loss))

    ax.axvline(x=epochs_ADAM, color='gray', linestyle='--',
               lw=1, alpha=0.5, label='Adam → L-BFGS')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('log(loss)')
    ax.set_title('Comparación de arquitecturas — NS cilindro con cilindro')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(root + 'loss_comparacion_todos.pdf', format='pdf')
    plt.show()

    # Tabla resumen
    print(f"\n{'Experimento':<22} {'Épocas':>7} {'Loss final':>12}")
    print("─" * 44)
    for name, epochs, loss in sorted(summary_rows, key=lambda r: r[2]):
        print(f"{name:<22} {epochs:>7} {loss:>12.4e}")

## 15. Inferencia espacial — modelo activo


In [ ]:
x_test, y_test = get_test_grid(nx=220, ny=80)
x_test_input   = torch.hstack((x_test, y_test)).to(device)

with torch.no_grad():
    preds = model(x_test_input).cpu().numpy()

x_np = x_test[:, 0].numpy()
y_np = y_test[:, 0].numpy()

fields = {
    r'$u^{(x)}$ — Velocidad en x' : preds[:, 0],
    r'$u^{(y)}$ — Velocidad en y' : preds[:, 1],
    r'$P$ — Presión'               : preds[:, 2],
}
fnames = ['ux', 'uy', 'p']

for (title, field), fname in zip(fields.items(), fnames):
    fig, ax = plt.subplots(figsize=(20, 3))
    sc = ax.scatter(x_np, y_np, c=field, cmap='jet', s=3)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(title.split('—')[0].strip(), fontsize=13,
                   rotation=0, labelpad=15)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(f'{title} — {experiment_name}')
    plt.tight_layout()
    plt.savefig(root + f'{fname}_{experiment_name}.pdf', format='pdf')
    plt.show()

## 15.1 Comparación espacial lado a lado — todos los modelos guardados


In [ ]:

model_registry = {
    'NS-FCN-BL'   : lambda: FCN(2, 50, 4, 3),
    'NS-PIKAN-S'  : lambda: PIKAN(2, 16, 2, 3, degree=3),
    'NS-PIKAN-M'  : lambda: PIKAN(2, 32, 3, 3, degree=5),
    'NS-PIKAN-H'  : lambda: PIKAN(2, 32, 3, 3, degree=7),
    'NS-cPIKAN-S' : lambda: cPIKAN(2, 16, 2, 3, degree=3),
    'NS-cPIKAN-M' : lambda: cPIKAN(2, 32, 3, 3, degree=5),
    'NS-cPIKAN-H' : lambda: cPIKAN(2, 32, 3, 3, degree=7),
}

# Malla de evaluación compartida — misma para todos los modelos
n_test = 300
x_test, y_test = get_collocation_points(
    n_test, circle_center_x, circle_center_y, circle_radius
)
x_test_input = torch.hstack((x_test, y_test)).to(device)
x_np = x_test_input[:, 0].cpu().detach().numpy()
y_np = x_test_input[:, 1].cpu().detach().numpy()

# Cargar predicciones de cada checkpoint disponible
loaded = {}   # {experiment_name: array (N,3)}
for fname in sorted(os.listdir(root)):
    if not fname.endswith('_checkpoint.pth'):
        continue
    ckpt = torch.load(root + fname, map_location=device)
    name = ckpt['experiment_name']
    if name not in model_registry:
        print(f"  ⚠ {name} no está en model_registry — saltando.")
        continue
    m = model_registry[name]()
    m.load_state_dict(ckpt['model_state'])
    m.to(device).eval()
    with torch.no_grad():
        pred = m(x_test_input).cpu().numpy()
    loaded[name] = pred
    print(f"  ✓ {name} cargado")

if not loaded:
    print("No hay checkpoints cargados. Verificar root y nombres.")
else:
    names      = list(loaded.keys())
    n_models   = len(names)
    field_labels = [
        (0, r'$u^{(x)}$', 'Velocidad en x'),
        (1, r'$u^{(y)}$', 'Velocidad en y'),
        (2, r'$P$',        'Presión'),
    ]

    for field_idx, cbar_label, field_title in field_labels:
        # Escala de color común entre todos los modelos
        all_vals = np.concatenate([loaded[n][:, field_idx] for n in names])
        vmin, vmax = all_vals.min(), all_vals.max()

        fig, axes = plt.subplots(
            1, n_models,
            figsize=(7 * n_models, 3),
            constrained_layout=True
        )
        if n_models == 1:
            axes = [axes]

        for ax, name in zip(axes, names):
            sc = ax.scatter(
                x_np, y_np,
                c=loaded[name][:, field_idx],
                cmap='jet', s=5,
                vmin=vmin, vmax=vmax
            )
            ax.set_title(name, fontsize=10)
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_aspect('equal')

        # Colorbar compartida al lado derecho
        cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
        cbar.set_label(cbar_label, fontsize=13, rotation=0, labelpad=15)
        fig.suptitle(f'{field_title} — comparación de arquitecturas',
                     fontsize=13, y=1.02)

        safe_title = field_title.replace(' ', '_').lower()
        plt.savefig(root + f'comparacion_{safe_title}.pdf',
                    format='pdf', bbox_inches='tight')
        plt.show()